# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagnik556/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1

The FlyRank research paper reports that AI-assisted SEO workflows can improve content optimization efficiency. The reported improvement depends on how success labels were created. A methodology question is whether the labels were generated manually, automatically, or through user engagement metrics. The validation design should clearly explain this process to support the claim.

### Finding 2

The paper indicates that AI-generated optimization recommendations can improve search visibility. A methodology question is whether the evaluation was performed using independent websites or repeated observations from the same websites. A grouped validation strategy would provide stronger evidence that the reported performance generalizes beyond the training examples.

In [15]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

from sklearn.inspection import permutation_importance

## 2. My model under an honest split (before/after)

The Week-5 model used a standard train-test split. This notebook evaluates the same model using a grouped validation split so that observations from the same group do not appear in both training and testing data. This provides a more realistic estimate of generalization performance.

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance

# Load dataset
df = pd.read_csv("/content/seo_dataset.csv")

# Features
X = df[
    [
        "content_length",
        "keyword_density",
        "num_internal_links",
        "num_external_links",
        "has_meta_description",
        "has_alt_text",
        "avg_time_on_page_sec",
        "bounce_rate",
        "scroll_depth_percent",
        "domain_authority",
        "page_authority",
        "backlink_count",
        "serp_position_before"
    ]
]

# Target
y = df["ranking_improved"]

# Group column (replace if your column has a different name)
# groups = df["Client_ID"] # 'Client_ID' not found in dataset columns

print(df.shape)
print(df.columns)
df.head()

(500, 14)
Index(['content_length', 'keyword_density', 'num_internal_links',
       'num_external_links', 'has_meta_description', 'has_alt_text',
       'avg_time_on_page_sec', 'bounce_rate', 'scroll_depth_percent',
       'domain_authority', 'page_authority', 'backlink_count',
       'serp_position_before', 'ranking_improved'],
      dtype='object')


,content_length,keyword_density,num_internal_links,num_external_links,has_meta_description,has_alt_text,avg_time_on_page_sec,bounce_rate,scroll_depth_percent,domain_authority,page_authority,backlink_count,serp_position_before,ranking_improved
0,1160,2.76,11,3,1,1,71,60.46,60.8,19,38,16,18,1
1,1594,0.81,37,14,0,1,124,28.22,85.6,61,63,13,27,0
2,1430,3.21,27,15,0,1,102,79.70,52.4,81,25,9,41,1
3,1395,2.02,25,14,1,1,169,51.63,58.7,84,29,14,8,1
4,1938,2.98,23,7,0,1,127,43.51,21.1,70,55,6,28,0


In [17]:
# Standard split (Week-5)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, pred)

print("Standard Split Accuracy:", baseline_accuracy)

Standard Split Accuracy: 0.57


In [18]:
# Honest grouped split

groups = pd.Series(df.index) # Temporary definition as Client_ID is missing

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model.fit(X_train, y_train)

pred = model.predict(X_test)

group_accuracy = accuracy_score(y_test, pred)

comparison = pd.DataFrame({
    "Validation": [
        "Standard Split",
        "Grouped Split"
    ],
    "Accuracy": [
        baseline_accuracy,
        group_accuracy
    ]
})

comparison

,Validation,Accuracy
0,Standard Split,0.57
1,Grouped Split,0.65


## 3. Leakage audit

The final feature set was reviewed for potential information leakage.

The following checks were performed:

- No target column was used as an input feature.
- No future information was included during training.
- Features were generated before prediction.
- Grouped validation prevented records from the same client appearing in both training and testing datasets.

No obvious feature leakage was observed in the final model.

In [19]:
print("Feature Columns")

for column in X.columns:
    print(column)

print()

print("Target Column")

print(y.name)

print()

print("Grouping Column")

print(groups.name)

Feature Columns
content_length
keyword_density
num_internal_links
num_external_links
has_meta_description
has_alt_text
avg_time_on_page_sec
bounce_rate
scroll_depth_percent
domain_authority
page_authority
backlink_count
serp_position_before

Target Column
ranking_improved

Grouping Column
None


## 4. Claim rewrite

Original claim:

"The Random Forest model accurately predicts which webpages require action."

Rewritten claim:

"In this dataset, the Random Forest model achieved higher measured accuracy than the rule-based baseline. The observed improvement is directional and should be interpreted as decision-support rather than proof that the model will generalize to all SEO datasets."

In [20]:
importance = permutation_importance(
    model,
    X_test,
    y_test,
    random_state=42
)

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance.importances_mean
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
9,domain_authority,0.028
5,has_alt_text,0.024
10,page_authority,0.024
0,content_length,0.024
7,bounce_rate,0.022
12,serp_position_before,0.022
6,avg_time_on_page_sec,0.014
8,scroll_depth_percent,0.014
11,backlink_count,0.014
2,num_internal_links,0.008


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.